# **SNU AI Challenge code

## **Task**

주어진 스토리라인(캡션)에 맞게 4개의 이미지 프레임을 올바른 순서로 재배열하는 문제를 해결해야 합니다.

- **Input**: 자연어 문장과 여러 장의 프레임으로 구성된 데이터
  - 예: `{ "text": "자연어 문장", "frames": [image_3, image_1, image_4, image_2] }`
- **Output**: 정답 순서대로 다시 배열하였을 때 각 프레임의 위치
  - 예: `[3, 4, 1, 2]`
  - 정답 순서대로 다시 배열하였을 때 첫 번째 프레임은 3번째에 위치, 두 번째 프레임은 4번째에 위치, …

## **Overall Pipeline**

| 단계 | 내용 |
|------|------|
| Step 1 | 데이터 로드 (test.csv + 이미지) |
| Step 2 | 모델 로드 (Qwen2-VL-2B-Instruct from HuggingFace) |
| Step 3 | 추론 진행 (프롬프트 → 모델 생성 → 출력 파싱) |
| Step 4 | 결과 저장 (submission.csv) |


## 0. 환경 설정 및 패키지 설치


In [1]:
%pip install -q torch torchvision transformers pandas pillow tqdm accelerate peft datasets
%pip install -q open_clip_torch
%pip install -q transformers accelerate qwen-vl-utils pillow pandas tqdm scikit-learn

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import ast
import re
import gc
import pandas as pd
import torch

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

In [3]:
#경로 설정
PROJECT_DIR = r"C:\SNU_2026"

DATA_DIR = os.path.join(PROJECT_DIR, "data")
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")

TRAIN_IMAGE_DIR = os.path.join(DATA_DIR, "train")
TEST_IMAGE_DIR = os.path.join(DATA_DIR, "test")

OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission_qwen3vl4b_enhanced.csv")

print("TRAIN_CSV:", TRAIN_CSV)
print("TEST_CSV:", TEST_CSV)
print("TRAIN_IMAGE_DIR:", TRAIN_IMAGE_DIR)
print("TEST_IMAGE_DIR:", TEST_IMAGE_DIR)
print("SUBMISSION_PATH:", SUBMISSION_PATH)

TRAIN_CSV: C:\SNU_2026\data\train.csv
TEST_CSV: C:\SNU_2026\data\test.csv
TRAIN_IMAGE_DIR: C:\SNU_2026\data\train
TEST_IMAGE_DIR: C:\SNU_2026\data\test
SUBMISSION_PATH: C:\SNU_2026\outputs\submission_qwen3vl4b_enhanced.csv


In [4]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print("train:", train_df.shape)
print("test:", test_df.shape)

display(train_df.head())
display(test_df.head())

train: (9535, 8)
test: (819, 6)


,Id,Input_1,Input_2,Input_3,Input_4,Sentence,Answer,No_ordering
0,diZi5g,diZi5g_ggm.jpg,diZi5g_jfd.jpg,diZi5g_nva.jpg,diZi5g_udj.jpg,The camera pans left to reveal two people knee...,"[1, 2, 3, 4]",True
1,u7w0lr,u7w0lr_aot.jpg,u7w0lr_shc.jpg,u7w0lr_uyv.jpg,u7w0lr_vuj.jpg,A girl hula hoops indoors before the scene shi...,"[3, 1, 2, 4]",False
2,2vqGOF,2vqGOF_buy.jpg,2vqGOF_nvf.jpg,2vqGOF_rlp.jpg,2vqGOF_vad.jpg,The woman lowers her gaze to apply a yellow co...,"[1, 2, 3, 4]",True
3,kSE41E,kSE41E_mug.jpg,kSE41E_ndb.jpg,kSE41E_nkm.jpg,kSE41E_vpx.jpg,"The man moves closer to the mirror, tilting hi...","[3, 2, 4, 1]",False
4,oO5UVE,oO5UVE_aat.jpg,oO5UVE_iox.jpg,oO5UVE_mkz.jpg,oO5UVE_qqu.jpg,The cookie sheet with dough moves closer to th...,"[3, 1, 2, 4]",False


,Id,Input_1,Input_2,Input_3,Input_4,Sentence
0,WI8o3F,WI8o3F_bia.jpg,WI8o3F_tuc.jpg,WI8o3F_vyx.jpg,WI8o3F_zcv.jpg,A person on an inner tube slides down a snowy ...
1,Ae7zLm,Ae7zLm_iis.jpg,Ae7zLm_otu.jpg,Ae7zLm_vqz.jpg,Ae7zLm_wru.jpg,The camera moves from a close-up of colorful n...
2,4QSIII,4QSIII_ayw.jpg,4QSIII_jqa.jpg,4QSIII_mqo.jpg,4QSIII_sto.jpg,The players on the beach court move forward an...
3,k7h4gt,k7h4gt_cuk.jpg,k7h4gt_ixd.jpg,k7h4gt_lqh.jpg,k7h4gt_rvh.jpg,The violin shifts from horizontal to vertical ...
4,ZtTwCx,ZtTwCx_npt.jpg,ZtTwCx_slc.jpg,ZtTwCx_whe.jpg,ZtTwCx_zve.jpg,"The person speaks to the camera, then lifts pa..."


In [5]:
#GPU 확인 및 모델 로드
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU를 사용할 수 없습니다.")

device = "cuda"

print("device:", device)
print("GPU:", torch.cuda.get_device_name(0))

gc.collect()
torch.cuda.empty_cache()

os.environ["HF_HUB_DISABLE_XET"] = "1"

MODEL_NAME = "Qwen/Qwen3-VL-4B-Instruct"

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)

model.eval()

print("model loaded")
print("device:", next(model.parameters()).device)

device: cuda
GPU: NVIDIA GeForce RTX 5070 Laptop GPU


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


model loaded
device: cuda:0


In [6]:
#정답 변환 함수
def parse_answer(answer):
    if isinstance(answer, str):
        return ast.literal_eval(answer)
    return answer


def order_to_submission_answer(order):
    """
    모델 출력: 시간순 이미지 번호
    예: [4, 2, 1, 3]

    제출 형식:
    Image 1이 몇 번째인지, Image 2가 몇 번째인지 ...
    예: [3, 2, 4, 1]
    """
    answer = [0] * 4

    for position, image_num in enumerate(order):
        answer[image_num - 1] = position + 1

    return answer


def parse_model_output(output_text):
    """
    모델 출력에서 [1, 2, 3, 4] 형태의 리스트만 추출합니다.
    이 값은 '시간순 이미지 번호'라고 가정합니다.
    """
    match = re.search(r"\[[^\]]+\]", output_text)

    if match is None:
        return [1, 2, 3, 4]

    try:
        pred_order = ast.literal_eval(match.group(0))
        pred_order = [int(x) for x in pred_order]

        if len(pred_order) == 4 and sorted(pred_order) == [1, 2, 3, 4]:
            return pred_order

    except Exception:
        pass

    return [1, 2, 3, 4]

In [7]:
#강화 프롬프트
def make_qwen_messages(row, image_dir):
    img_files = [
        row["Input_1"],
        row["Input_2"],
        row["Input_3"],
        row["Input_4"],
    ]

    content = []

    for i, img_file in enumerate(img_files):
        img_path = os.path.join(image_dir, str(row["Id"]), str(img_file))

        content.append({
            "type": "image",
            "image": img_path,
        })

        content.append({
            "type": "text",
            "text": f"\nThis is Image {i + 1}.\n",
        })

    sentence = str(row["Sentence"])

    prompt = f"""
The 4 images above are SHUFFLED video frames.
They are NOT guaranteed to be in chronological order.

Sentence describing the event:
"{sentence}"

Your task:
Find the correct chronological order of Image 1, Image 2, Image 3, and Image 4.

Important:
- Do NOT assume the input image order is correct.
- Use only visual evidence from the images.
- Focus on what happens before and after.
- Compare hand movement, object movement, body pose, scene changes, and action progress.
- The answer must be the order of image numbers from earliest to latest.

Return ONLY a Python list of 4 integers.
Use each number 1, 2, 3, 4 exactly once.
Do not write any explanation.

Correct output format example:
[4, 2, 1, 3]
""".strip()

    content.append({
        "type": "text",
        "text": prompt,
    })

    return [
        {
            "role": "user",
            "content": content,
        }
    ]

In [11]:
import itertools
from collections import Counter

# 4개 이미지 순서의 모든 permutation
ALL_PERMS = list(itertools.permutations([0, 1, 2, 3]))

def make_qwen_messages_tta(row, image_dir, perm):
    """
    perm 순서대로 이미지를 모델에 보여줍니다.
    예: perm = (2, 0, 1, 3)이면
    Image 1 = 원래 Input_3
    Image 2 = 원래 Input_1
    Image 3 = 원래 Input_2
    Image 4 = 원래 Input_4
    """
    img_files = [
        row["Input_1"],
        row["Input_2"],
        row["Input_3"],
        row["Input_4"],
    ]

    content = []

    for display_idx, original_idx in enumerate(perm):
        img_file = img_files[original_idx]
        img_path = os.path.join(image_dir, str(row["Id"]), str(img_file))

        content.append({
            "type": "image",
            "image": img_path,
        })

        content.append({
            "type": "text",
            "text": f"\nImage {display_idx + 1}\n",
        })

    sentence = str(row["Sentence"])

    prompt = f"""
The 4 images above are shuffled video frames.
They are NOT guaranteed to be in chronological order.

Sentence:
{sentence}

Your task is to determine the correct chronological order of the images.

Important rules:
- Do NOT assume the input image order is correct.
- Compare action progress, object movement, body posture, and scene changes.
- Return ONLY a Python list of 4 integers.
- Each integer must be one of 1, 2, 3, 4.
- Use each integer exactly once.
- Do not explain.

Example output:
[2, 1, 4, 3]
""".strip()

    content.append({
        "type": "text",
        "text": prompt,
    })

    return [
        {
            "role": "user",
            "content": content,
        }
    ]

In [12]:
#예측함수
def predict_qwen_order(row, image_dir):
    messages = make_qwen_messages(row, image_dir)

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    inputs = inputs.to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    pred_order = parse_model_output(output_text)
    pred_answer = order_to_submission_answer(pred_order)

    return pred_order, pred_answer, output_text

In [15]:
import ast
import re

def parse_order_output(output_text):
    """
    Qwen3-VL이 출력한 텍스트에서 [1, 2, 3, 4] 같은 리스트를 찾아서 반환합니다.

    반환값은 chronological order입니다.
    예:
    raw output: "The answer is [3, 1, 2, 4]"
    return: [3, 1, 2, 4]
    """
    output_text = str(output_text).strip()

    # 먼저 [1, 2, 3, 4] 형태를 찾습니다.
    match = re.search(r"\[[^\]]+\]", output_text)

    if match:
        try:
            result = ast.literal_eval(match.group())

            if (
                isinstance(result, list)
                and len(result) == 4
                and sorted(result) == [1, 2, 3, 4]
            ):
                return result

        except Exception:
            pass

    # 혹시 대괄호 없이 "1, 2, 3, 4" 또는 "1 2 3 4"처럼 출력한 경우도 처리합니다.
    numbers = [int(x) for x in re.findall(r"\b[1-4]\b", output_text)]

    if len(numbers) >= 4:
        result = numbers[:4]

        if sorted(result) == [1, 2, 3, 4]:
            return result

    # 파싱 실패 시 기본값
    return [1, 2, 3, 4]

In [16]:
def predict_qwen_order_tta(row, image_dir, perms=None):
    """
    여러 이미지 입력 순서로 Qwen3-VL 추론을 실행한 뒤,
    원래 이미지 번호 기준 예측으로 되돌리고 majority vote를 합니다.
    """
    if perms is None:
        # 전체 24개 permutation은 느리므로 처음에는 6개만 사용
        perms = [
            (0, 1, 2, 3),
            (1, 0, 2, 3),
            (2, 1, 0, 3),
            (3, 1, 2, 0),
            (0, 2, 1, 3),
            (0, 3, 2, 1),
        ]

    original_order_predictions = []
    raw_outputs = []

    for perm in perms:
        messages = make_qwen_messages_tta(row, image_dir, perm)

        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        image_inputs, video_inputs = process_vision_info(messages)

        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )

        inputs = inputs.to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=32,
                do_sample=False,
                use_cache=True,
            )

        generated_ids_trimmed = [
            out_ids[len(in_ids):]
            for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]

        raw_text = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]

        raw_outputs.append(raw_text)

        pred_display_order = parse_order_output(raw_text)

        # 모델 출력은 "현재 보여준 Image 번호 기준"입니다.
        # 이것을 "원래 Input_1~Input_4 기준"으로 되돌립니다.
        pred_original_order = [
            perm[display_num - 1] + 1
            for display_num in pred_display_order
        ]

        original_order_predictions.append(tuple(pred_original_order))

    # 가장 많이 나온 순서를 최종 예측으로 사용
    vote_counter = Counter(original_order_predictions)
    best_order = list(vote_counter.most_common(1)[0][0])

    return best_order, raw_outputs, vote_counter

In [18]:
from sklearn.model_selection import train_test_split

# train.csv를 학습용/검증용으로 나눕니다.
# test.csv에는 정답이 없기 때문에 validation은 train.csv에서 직접 만들어야 합니다.
train_part, valid_part = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["Answer"],
)

# Qwen3 추론은 느리기 때문에 먼저 작은 검증 데이터로 빠르게 테스트합니다.
# 성능이 괜찮으면 50 -> 100 -> 300처럼 늘려가면 됩니다.
valid_small = valid_part.sample(
    n=50,
    random_state=42
).reset_index(drop=True)

print("train_part:", train_part.shape)
print("valid_part:", valid_part.shape)
print("valid_small:", valid_small.shape)

correct = 0
total = 0
identity_count = 0
bad_outputs = []

for idx, row in tqdm(valid_small.iterrows(), total=len(valid_small)):
    pred_order, raw_outputs, vote_counter = predict_qwen_order_tta(
        row,
        TRAIN_IMAGE_DIR,
    )

    # chronological order를 competition answer 형식으로 변환
    pred_answer = order_to_submission_answer(pred_order)
    true_answer = parse_answer(row["Answer"])

    if pred_answer == true_answer:
        correct += 1

    if pred_order == [1, 2, 3, 4]:
        identity_count += 1

    total += 1

accuracy = correct / total
identity_ratio = identity_count / total

print("valid_small accuracy:", accuracy)
print("identity ratio:", identity_ratio)

train_part: (7628, 8)
valid_part: (1907, 8)
valid_small: (50, 8)


  0%|          | 0/50 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [10]:
#Test 추론
predictions = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    pred_order, pred_answer, raw = predict_qwen_order(row, TEST_IMAGE_DIR)

    predictions.append({
        "Id": row["Id"],
        "Answer": str(pred_answer),
    })

submission_df = pd.DataFrame(predictions)

print("submission:", submission_df.shape)
display(submission_df.head())

  0%|          | 0/819 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 4. 결과 저장 (Submission)

### 제출 파일 형식 (submission.csv)

제출 파일은 반드시 아래 형식을 따라야 합니다:

| 컬럼 | 타입 | 설명 | 예시 |
|------|------|------|------|
| `Id` | string | 테스트 샘플 고유 ID | `WI8o3F` |
| `Answer` | string | 정답 순서대로 다시 배열하였을 때 각 프레임의 위치 (Python 리스트 형태의 문자열) | `[3, 1, 4, 2]` |

#### 올바른 예시

```csv
Id,Answer
WI8o3F,[3, 1, 4, 2]
Ae7zLm,[1, 2, 3, 4]
...


In [ ]:
#제출 파일 저장
submission_df = test_df[["Id"]].merge(
    submission_df,
    on="Id",
    how="left",
)

submission_df["Answer"] = submission_df["Answer"].fillna(str([1, 2, 3, 4]))

submission_df.to_csv(SUBMISSION_PATH, index=False)

print("saved:", SUBMISSION_PATH)
print("submission shape:", submission_df.shape)

display(submission_df.head())